# PI-CAM-only: Python control with original iCESM CAM numerics

This notebook is intentionally separate from the CAM-SIMA examples. The full PI-atm executable supplies only the captured boundary oracle; the runtime shown here starts CAM alone on 512 MPI ranks. A generated state bridge creates and initializes active `cam_in`, `cam_out`, `phys_state`, and `phys_tend` numeric members as rank-local Python-owned NumPy arrays before native CAM resumes.

In [ ]:
from datetime import datetime
from pathlib import Path
import json
import numpy as np
import os
import subprocess

from freecam.pi_cam import (
    PICAMCase,
    PICAMNotebookSession,
    PICAMStepPlan,
)

repo = Path('/glade/work/ruitong/freeCAM')
config_path = repo / 'configs/pi_cam_icesm131.yaml'
case = PICAMCase.from_yaml(config_path)
case.config.to_payload()

## Python owns the exact CAM action order

This is inspection only: it does not initialize Fortran or MPI. The first boundary action is followed by the granular `cam_run2`, `cam_run3`, `cam_run4`, clock, and `cam_run1` actions, then the CAM export.

In [ ]:
PICAMStepPlan.default().describe()

## Run the real 512-rank CAM-only model on cpudev

The maintained job uses one `mpiexec -n 512`, replayed rank-local CAM inputs, the fixed-address non-PIC CAM `.so`, and a fail-closed comparison against the original oracle. Set `submit = True` only when you want a new run.

In [ ]:
submit = False
job = repo / 'validation/jobs/pi_cam_python_zero_copy_state_50step.pbs'
if submit:
    submitted_job = subprocess.run(
        ['qsub', str(job)], check=True, capture_output=True, text=True
    ).stdout.strip()
    print('submitted:', submitted_job)
else:
    print('Not submitted; set submit = True to run the 512-rank gate.')

## Inspect the completed scientific gate

`bfb: true` means every numeric CAM history/restart variable has identical bits. Character metadata such as timestamps and run-directory paths is intentionally excluded.

In [ ]:
evidence = repo / 'validation/pi_cam_python_zero_copy_state_vs_oracle_50step_bfb.json'
json.loads(evidence.read_text()) if evidence.exists() else {
    'status': 'Run the previous cell and wait for the PBS job to finish.'
}

## Start one persistent 512-rank model for interactive cells

The session submits one cpudev job and performs one `mpiexec -n 512`. The following cells reuse the same live CAM memory. Closing the session is the only operation that ends those ranks.

In [ ]:
scratch = Path(os.environ.get('SCRATCH', '/glade/derecho/scratch/ruitong'))
oracle_case = Path(
    '/glade/work/ruitong/CESM_cases/'
    'f.e13.F1850C5.ne16_g16.icesm131_ihesp.PI-cam-oracle.50step'
)
oracle_run = scratch / (
    'pyCAM/PI-cam/'
    'f.e13.F1850C5.ne16_g16.icesm131_ihesp.PI-cam-oracle.50step/run'
)
replay_root = scratch / (
    'pyCAM/PI-cam/nonpic-boundary-capture-50step/boundary/replay'
)
stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
interactive_run = scratch / f'pyCAM/PI-cam/notebook-{stamp}/run'
interactive_run.mkdir(parents=True, exist_ok=False)
subprocess.run([
    'rsync', '-a',
    '--exclude', '*.cam.*.nc', '--exclude', '*.log.*',
    '--exclude', 'rpointer.*', '--exclude', 'timing/',
    f'{oracle_run}/', f'{interactive_run}/',
], check=True)
(interactive_run / 'timing/checkpoints').mkdir(parents=True)
interactive_run

In [ ]:
cam = PICAMNotebookSession(
    config_path,
    boundary=replay_root,
    run_dir=interactive_run,
    env_script=oracle_case / '.env_mach_specific.sh',
    python_executable=repo / '.venv/bin/python',
)
cam.start()
cam.status

## Reuse the same live StatePool across cells

`step()` executes one complete source-ordered CAM step. `field()` copies one selected rank-local NumPy array back to Jupyter; `stats(..., rank='global')` reduces compact statistics across all 512 ranks. The generated bridge discovers 136 numeric legacy fields, creates the 134 active fields in each rank's StatePool, and binds CAM to the same addresses without a mirror copy.

In [ ]:
cam.step()
rank0_export = cam.field('cam_out.a2x_rattr', rank=0)
rank0_temperature = cam.field('phys_state.t', rank=0)
rank0_ncol = cam.field('phys_state.ncol', rank=0)
active_temperature = np.concatenate([
    rank0_temperature[:int(rank0_ncol[chunk]), :, chunk]
    for chunk in range(rank0_temperature.shape[-1])
])
{
    'status': cam.status,
    'rank0_export_shape': rank0_export.shape,
    'rank0_physics_temperature_shape': rank0_temperature.shape,
    'global_export_stats': cam.stats('cam_out.a2x_rattr', rank='global'),
    'rank0_active_physics_temperature_min': float(active_temperature.min()),
    'rank0_active_physics_temperature_max': float(active_temperature.max()),
}

## Optional fine-grained experiments

These calls intentionally bypass normal dependencies, so use a disposable session. They do not advance model time automatically.

In [ ]:
# Run only one original Fortran physics scheme:
dadadj_trace = cam.run_scheme('dadadj', phase='cam_run1')

# Run only the dadadj leaf routine. Its two compact grid metadata arrays
# and five physics StatePool arrays are passed directly to the adapter.
# The adapter then calls the original dadadj_ symbol.
# This is not the complete dry-adjustment host stage above.
dadadj_kernel_trace = cam.run_kernel('dadadj')

# Or run every enabled action in one CAM phase:
cam_run3_trace = cam.run_phase('cam_run3')
dadadj_trace, dadadj_kernel_trace, cam_run3_trace

## Release the persistent MPI job

Run this cell when finished. It calls CAM finalize, exits all 512 ranks, and releases the PBS allocation.

In [ ]:
cam.close()